# Eardrum demo

Dual-head onsets-and-frames transcription on a 20 s MAESTRO excerpt.
Run `python train.py` and `python evaluate.py` first so `outputs/dual_best.pt`
and `outputs/metrics.json` exist, then run this notebook top to bottom from the
repository root.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

from app import piano_roll_figure
from eardrum import constants as C
from eardrum.dataset import load_audio
from eardrum.decode import decode_notes, load_model, notes_to_midi, predict_probabilities

CHECKPOINT = Path("outputs/dual_best.pt")
AUDIO = sorted(Path("outputs/demo_samples").glob("*.wav"))[0]
model, _ = load_model(CHECKPOINT)
audio = load_audio(AUDIO)
print(f"{AUDIO.name}: {len(audio) / C.SAMPLE_RATE:.1f} s")

In [ ]:
probs = predict_probabilities(model, audio)
notes = decode_notes(probs["frame"], probs["onset"], onset_thresh=0.45, frame_thresh=0.40)
midi = notes_to_midi(notes, duration=len(audio) / C.SAMPLE_RATE)
print(f"{len(notes)} notes")
if notes:
    pitches = [n[0] for n in notes]
    print(f"pitch range {min(pitches)}–{max(pitches)} (MIDI)")

In [ ]:
figure = piano_roll_figure(audio, probs["frame"], frame_thresh=0.40)
plt.show()

In [ ]:
waveform = midi.synthesize(fs=C.SAMPLE_RATE)
if waveform.size:
    display(Audio(waveform, rate=C.SAMPLE_RATE))
else:
    print("No notes detected; try lowering the thresholds.")

In [ ]:
midi.write("outputs/demo_transcription.mid")
print("wrote outputs/demo_transcription.mid")

Test-set scores for both models are in `outputs/metrics.json`; the three report
figures are `ablation_bar_chart.png`, `qualitative_pianoroll.png`, and
`threshold_sweep.png` in `outputs/`.